# Behavioral Decision-Making Model
**Fixes applied:**
- `clicked_item` removed from features (was causing data leakage)
- `brochure_id` now treated as categorical (one-hot encoded)
- 5-fold cross-validation added for reliable accuracy estimates
- Real participant data accumulation: appends to `real_data.csv` each run
- Model trains on real data if available (≥10 rows), otherwise falls back to synthetic
- SHAP feature importance added for XAI
- Age/gender collected once via Flask API call (no blocking `input()` in production)

In [ ]:
import ast
import os
import glob
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Optional SHAP — install with: pip install shap
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed. Run: pip install shap  to enable XAI output.")

In [ ]:
# =========================================================
# CONFIG — adjust paths if needed
# =========================================================
DATA_FOLDER    = r"C:\Users\M S I\OneDrive\Documents\GitHub\Research-Project\data"
TRAIN_FILE     = os.path.join(DATA_FOLDER, "synthetic_dataset_with_item.csv")
REAL_DATA_FILE = os.path.join(DATA_FOLDER, "real_participant_data.csv")   # accumulated real data
MODEL_FILE     = os.path.join(DATA_FOLDER, "reason_prediction_model.pkl")
CONVERTED_FILE = os.path.join(DATA_FOLDER, "latest_converted_psychopy.csv")
PREDICTION_FILE= os.path.join(DATA_FOLDER, "latest_reason_percentages.csv")
RESULTS_JSON   = os.path.join(DATA_FOLDER, "latest_results.json")         # read by Flask backend

# Minimum real rows before switching away from synthetic data
REAL_DATA_THRESHOLD = 10

In [ ]:
# =========================================================
# HELPERS
# =========================================================
def parse_list_cell(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, (int, float)):
        return [value]
    value = str(value).strip()
    if value == "" or value.lower() == "nan":
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else [parsed]
    except Exception:
        return [value]


def get_last_numeric(value):
    nums = []
    for v in parse_list_cell(value):
        try:
            nums.append(float(v))
        except Exception:
            pass
    return nums[-1] if nums else np.nan


def get_last_string(value):
    arr = parse_list_cell(value)
    return str(arr[-1]).strip() if arr else None


def standardize_reason_labels(series):
    return series.astype(str).str.strip().str.lower().replace(
        {"price": "Price", "brand": "Brand", "familiarity": "Familiarity"}
    )


def normalize_gender(value):
    gender_map = {"male": "Male", "m": "Male", "female": "Female", "f": "Female"}
    return gender_map.get(str(value).strip().lower(), str(value).strip().title())


def get_user_age():
    while True:
        try:
            age = int(input("Enter participant age: ").strip())
            if age > 0:
                return age
            print("Please enter a valid positive age.")
        except ValueError:
            print("Invalid input. Please enter a whole number for age.")


def get_user_gender():
    while True:
        gender = normalize_gender(input("Enter participant gender (Male/Female): ").strip())
        if gender in ["Male", "Female"]:
            return gender
        print("Invalid input. Please enter Male or Female.")

In [ ]:
# =========================================================
# FIND LATEST PSYCHOPY CSV
# =========================================================
def get_latest_psychopy_csv(folder_path):
    skip_keywords = {"synthetic", "converted", "prediction", "latest_reason", "real_participant"}
    valid_files = [
        f for f in glob.glob(os.path.join(folder_path, "*.csv"))
        if not any(kw in os.path.basename(f).lower() for kw in skip_keywords)
    ]
    if not valid_files:
        raise FileNotFoundError("No valid PsychoPy CSV files found in the folder.")
    return max(valid_files, key=os.path.getmtime)

In [ ]:
# =========================================================
# BUILD SKLEARN PIPELINE
# FIX 1: brochure_id is now CATEGORICAL (one-hot encoded)
# FIX 2: clicked_item REMOVED (was data leakage)
# =========================================================
def build_pipeline():
    # Purely behavioral / demographic signals — no item identity
    numeric_features     = ["age", "reaction_time", "gaze_x", "gaze_y"]
    categorical_features = ["gender", "brochure_id"]   # brochure_id → one-hot

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler())
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer,     numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=2,
            random_state=42,
            class_weight="balanced"
        ))
    ])
    return model, numeric_features, categorical_features

In [ ]:
# =========================================================
# TRAIN MODEL
# FIX 3: uses real data when available, synthetic as fallback
# FIX 4: 5-fold cross-validation for reliable accuracy estimate
# =========================================================
def train_model(train_file, real_data_file, model_file):
    # Decide which data to train on
    use_real = False
    if os.path.exists(real_data_file):
        real_df = pd.read_csv(real_data_file)
        if len(real_df) >= REAL_DATA_THRESHOLD:
            df = real_df.copy()
            use_real = True
            print(f"Training on REAL participant data ({len(df)} rows).")

    if not use_real:
        df = pd.read_csv(train_file)
        print(f"Training on SYNTHETIC data ({len(df)} rows). "
              f"Collect ≥{REAL_DATA_THRESHOLD} real rows to switch automatically.")

    required_cols = ["brochure_id", "age", "gender", "reason", "reaction_time", "gaze_x", "gaze_y"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset missing columns: {missing}")

    df = df.copy()
    df["reason"]      = standardize_reason_labels(df["reason"])
    df["gender"]      = df["gender"].astype(str).str.strip().str.title()
    df["brochure_id"] = df["brochure_id"].astype(str)   # treat as category
    df = df.dropna(subset=["reason"])
    df = df[df["reason"].isin(["Brand", "Price", "Familiarity"])].reset_index(drop=True)

    feature_cols = ["brochure_id", "age", "gender", "reaction_time", "gaze_x", "gaze_y"]
    X = df[feature_cols].copy()
    y = df["reason"].copy()

    model, _, _ = build_pipeline()

    # --- 5-fold cross-validation (FIX 4) ---
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    print(f"\n5-Fold CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    print(f"CV Scores per fold: {[round(s, 3) for s in cv_scores]}")

    # --- Final train/test split for held-out report ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    test_acc = accuracy_score(y_test, y_pred)

    print("\n======================================")
    print("MODEL TRAINING RESULTS")
    print("======================================")
    print(f"Held-out Test Accuracy : {round(test_acc, 4)}")
    print(f"CV Mean Accuracy       : {round(cv_scores.mean(), 4)}")
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

    joblib.dump(model, model_file)
    print(f"Model saved to: {model_file}\n")

    return model, test_acc, cv_scores

In [ ]:
# =========================================================
# CONVERT RAW PSYCHOPY FILE TO TIDY FORMAT
# =========================================================
def convert_psychopy_to_tidy(psychopy_file, age_value, gender_value, output_file):
    raw_df = pd.read_csv(psychopy_file)
    if raw_df.empty:
        raise ValueError("The PsychoPy CSV file is empty.")

    participant_value = "Unknown"
    if "participant" in raw_df.columns:
        val = raw_df["participant"].dropna()
        if not val.empty:
            participant_value = str(val.iloc[0]).strip()

    brochure_blocks = [
        {"brochure_id": 1, "time_col": "mouse_5.time", "x_col": "mouse_5.x", "y_col": "mouse_5.y", "clicked_col": "mouse_5.clicked_name"},
        {"brochure_id": 2, "time_col": "mouse_4.time", "x_col": "mouse_4.x", "y_col": "mouse_4.y", "clicked_col": "mouse_4.clicked_name"},
        {"brochure_id": 3, "time_col": "mouse_3.time", "x_col": "mouse_3.x", "y_col": "mouse_3.y", "clicked_col": "mouse_3.clicked_name"},
        {"brochure_id": 4, "time_col": "mouse.time",   "x_col": "mouse.x",   "y_col": "mouse.y",   "clicked_col": "mouse.clicked_name"},
        {"brochure_id": 5, "time_col": "mouse_2.time", "x_col": "mouse_2.x", "y_col": "mouse_2.y", "clicked_col": "mouse_2.clicked_name"},
    ]

    tidy_rows = []
    for block in brochure_blocks:
        bid  = block["brochure_id"]
        c_col = block["clicked_col"]
        if c_col not in raw_df.columns:
            print(f"Skipping brochure {bid}: missing column {c_col}")
            continue
        valid_rows = raw_df[raw_df[c_col].notna()]
        if valid_rows.empty:
            print(f"Skipping brochure {bid}: no clicked item found")
            continue
        row = valid_rows.iloc[0]
        tidy_rows.append({
            "participant":   participant_value,
            "brochure_id":   str(bid),            # string for categorical encoding
            "age":           age_value,
            "gender":        gender_value,
            "reaction_time": get_last_numeric(row[block["time_col"]]) if block["time_col"] in raw_df.columns else np.nan,
            "gaze_x":        get_last_numeric(row[block["x_col"]])    if block["x_col"]   in raw_df.columns else np.nan,
            "gaze_y":        get_last_numeric(row[block["y_col"]])    if block["y_col"]   in raw_df.columns else np.nan,
            "clicked_item":  get_last_string(row[c_col])              # kept as metadata only
        })

    tidy_df = pd.DataFrame(tidy_rows).sort_values("brochure_id").reset_index(drop=True)
    if tidy_df.empty:
        raise ValueError("No valid brochure data extracted from PsychoPy CSV.")

    tidy_df.to_csv(output_file, index=False)
    print("\n=====================================")
    print("CONVERTED LATEST PSYCHOPY FILE")
    print("=====================================")
    print(tidy_df.to_string(index=False))
    print(f"\nConverted file saved to: {output_file}\n")
    return tidy_df

In [ ]:
# =========================================================
# ACCUMULATE REAL PARTICIPANT DATA (FIX 3)
# Appends each session's tidy rows to real_participant_data.csv
# Requires researcher to label 'reason' after each session
# =========================================================
def accumulate_real_data(tidy_df, real_data_file, reason_label=None):
    """
    Appends this session's rows to the real data accumulator.
    reason_label: if you already know the ground-truth reason for all rows,
                  pass it as a string (e.g. 'Brand'). Otherwise rows are saved
                  without a label and must be labeled manually before retraining.
    """
    df = tidy_df.copy()
    if reason_label:
        df["reason"] = reason_label
    else:
        df["reason"] = np.nan   # label manually in the CSV later

    if os.path.exists(real_data_file):
        existing = pd.read_csv(real_data_file)
        combined = pd.concat([existing, df], ignore_index=True)
    else:
        combined = df

    combined.to_csv(real_data_file, index=False)
    labeled = combined["reason"].notna().sum()
    print(f"Real data file updated: {len(combined)} total rows, {labeled} labeled.")
    if labeled < REAL_DATA_THRESHOLD:
        print(f"  → Need {REAL_DATA_THRESHOLD - labeled} more labeled rows to switch from synthetic training.")
    return combined

In [ ]:
# =========================================================
# PREDICT REASON PERCENTAGES + XAI (FIX: SHAP added)
# =========================================================
def predict_reason_percentages(tidy_df, model_file, output_file, results_json_file):
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"Model file not found: {model_file}")

    model = joblib.load(model_file)

    # Features — NO clicked_item (FIX 2)
    feature_cols = ["brochure_id", "age", "gender", "reaction_time", "gaze_x", "gaze_y"]
    X_new = tidy_df[feature_cols].copy()
    X_new["brochure_id"] = X_new["brochure_id"].astype(str)

    probs      = model.predict_proba(X_new)
    preds      = model.predict(X_new)
    class_names = model.named_steps["classifier"].classes_

    prob_df = pd.DataFrame(probs * 100, columns=[f"{c}_Percentage" for c in class_names])

    final_df = tidy_df.copy()
    final_df["Predicted_Reason"] = preds
    for col in prob_df.columns:
        final_df[col] = prob_df[col].round(2)

    final_df.to_csv(output_file, index=False)

    # --- SHAP feature importance ---
    shap_summary = []
    if SHAP_AVAILABLE:
        try:
            rf_clf     = model.named_steps["classifier"]
            preprocessor = model.named_steps["preprocessor"]
            X_transformed = preprocessor.transform(X_new)

            explainer  = shap.TreeExplainer(rf_clf)
            shap_vals  = explainer.shap_values(X_transformed)  # list per class

            # Get feature names after one-hot
            num_names = ["age", "reaction_time", "gaze_x", "gaze_y"]
            cat_names = list(preprocessor.named_transformers_["cat"]
                             .named_steps["onehot"].get_feature_names_out(["gender", "brochure_id"]))
            all_feature_names = num_names + cat_names

            # Mean absolute SHAP across classes and rows
            mean_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_vals], axis=0)
            shap_df   = pd.DataFrame({"feature": all_feature_names, "importance": mean_shap})
            shap_df   = shap_df.sort_values("importance", ascending=False).head(10)

            print("\n=====================================")
            print("SHAP FEATURE IMPORTANCE (top 10)")
            print("=====================================")
            print(shap_df.to_string(index=False))
            shap_summary = shap_df.to_dict(orient="records")
        except Exception as e:
            print(f"SHAP computation skipped: {e}")

    # --- Print reason influence report ---
    print("\n=====================================")
    print("PREDICTION OUTPUT")
    print("=====================================")
    print(final_df.to_string(index=False))

    print("\n=====================================")
    print("REASON INFLUENCE REPORT")
    print("=====================================")

    results_rows = []
    for _, row in final_df.iterrows():
        brand_pct  = row.get("Brand_Percentage", 0)
        fam_pct    = row.get("Familiarity_Percentage", 0)
        price_pct  = row.get("Price_Percentage", 0)

        print(f"\nParticipant   : {row['participant']}")
        print(f"Brochure ID   : {row['brochure_id']}")
        print(f"Clicked Item  : {row.get('clicked_item', 'N/A')}  [metadata only — not used as feature]")
        print(f"Age           : {row['age']}  |  Gender: {row['gender']}")
        rt = row['reaction_time']
        print(f"Reaction Time : {rt:.4f}s" if pd.notna(rt) else "Reaction Time : NaN")
        print(f"Gaze (X, Y)   : ({row['gaze_x']:.4f}, {row['gaze_y']:.4f})" if pd.notna(row['gaze_x']) else "Gaze: NaN")
        print(f"\nReason influence percentages:")
        print(f"   Brand       : {brand_pct:.2f}%")
        print(f"   Familiarity : {fam_pct:.2f}%")
        print(f"   Price       : {price_pct:.2f}%")
        print(f"\n→  Final Decision Most Influenced By : {row['Predicted_Reason']}")
        print("--------------------------------------")

        results_rows.append({
            "participant":    str(row["participant"]),
            "brochure_id":   row["brochure_id"],
            "clicked_item":  row.get("clicked_item", "N/A"),
            "age":           int(row["age"]),
            "gender":        row["gender"],
            "reaction_time": round(float(rt), 4) if pd.notna(rt) else None,
            "gaze_x":        round(float(row["gaze_x"]), 4) if pd.notna(row["gaze_x"]) else None,
            "gaze_y":        round(float(row["gaze_y"]), 4) if pd.notna(row["gaze_y"]) else None,
            "brand_pct":     round(float(brand_pct), 2),
            "familiarity_pct": round(float(fam_pct), 2),
            "price_pct":     round(float(price_pct), 2),
            "predicted_reason": row["Predicted_Reason"]
        })

    # --- Write JSON for Flask backend ---
    results_payload = {
        "predictions": results_rows,
        "shap_importance": shap_summary
    }
    with open(results_json_file, "w") as f:
        json.dump(results_payload, f, indent=2)
    print(f"\nResults JSON saved to: {results_json_file}")
    print(f"Prediction CSV saved to: {output_file}\n")

    return final_df, results_payload

In [ ]:
# =========================================================
# MAIN — run all steps
# =========================================================
if __name__ == "__main__":
    # 1. Collect participant metadata
    age_value    = get_user_age()
    gender_value = get_user_gender()

    # 2. Train (real data if available, else synthetic)
    model, test_acc, cv_scores = train_model(TRAIN_FILE, REAL_DATA_FILE, MODEL_FILE)

    # 3. Convert latest PsychoPy CSV
    latest_csv = get_latest_psychopy_csv(DATA_FOLDER)
    print(f"\nLatest PsychoPy CSV: {latest_csv}")
    tidy_df = convert_psychopy_to_tidy(latest_csv, age_value, gender_value, CONVERTED_FILE)

    # 4. Accumulate real data for future retraining
    #    Pass reason_label='Brand' if you know the ground truth, else leave None
    accumulate_real_data(tidy_df, REAL_DATA_FILE, reason_label=None)

    # 5. Predict + XAI + write JSON for Flask
    results_df, results_payload = predict_reason_percentages(
        tidy_df, MODEL_FILE, PREDICTION_FILE, RESULTS_JSON
    )